# 06. Feature Engineering Preparation

This notebook validates the metro and weather data structures, builds the citywide hourly ridership dataset, and merges it with the cleaned weather observations before any lag or rolling features are created.

The focus is on data integrity and timestamp alignment only. No lag features, rolling statistics, or model-ready engineered features are added in this step.

## 1. Existing metro aggregation logic and meaning of `count`

The metro source data in the raw Excel workbook is organised by sheet name: `month_direction` (for example, `september_2024_entry` and `september_2024_exit`). Each sheet contains one row per station/date and hourly columns such as `HR4`, `HR5`, `HR6`, and so on.

The existing notebook code melts these wide hourly columns into long format, producing one row per station/date/hour/direction. In that long format, `count` represents the hourly passenger count for a single station at a single hour in a single direction.

Therefore, `count` is not the total city demand; it is the station-level hourly flow, and citywide totals are obtained by summing station counts by `businessday`, `hour`, and `direction`.

In [2]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd().resolve()
if project_root.name == 'Notebooks':
    project_root = project_root.parent

metro_path = project_root / 'DataProcessed' / 'metro_master.csv'
weather_path = project_root / 'DataProcessed' / 'weather_hourly_cleaned.csv'
output_path = project_root / 'DataProcessed' / 'metro_weather_hourly_merged.csv'

metro = pd.read_csv(metro_path)
weather = pd.read_csv(weather_path, parse_dates=['time'])

print(f'Metro rows: {len(metro):,}')
print(f'Weather rows: {len(weather):,}')
print('Metro columns:', list(metro.columns))
print('Weather columns:', list(weather.columns))
print('\nMetro sample:')
display(metro.head())
print('\nWeather sample:')
display(weather.head())

Metro rows: 1,872,465
Weather rows: 4,344
Metro columns: ['businessday', 'linename', 'sitename', 'station name', 'station code', 'hour', 'count', 'month', 'direction']
Weather columns: ['time', 'temperature_2m_c', 'relative_humidity_2m_pct', 'apparent_temperature_c', 'precipitation_mm', 'rain_mm', 'weather_code_wmo', 'surface_pressure_hpa', 'cloud_cover_pct', 'visibility_m', 'wind_speed_10m_kmh']

Metro sample:


,businessday,linename,sitename,station name,station code,hour,count,month,direction
0,2024-09-01,RMGL,Sector 55-56-RMGL,Sector 55-56,Sector 55-56,4,0,September,Entry
1,2024-09-01,LINE06,Nehru Place,Nehru Place,NP,4,0,September,Entry
2,2024-09-01,LINE06,Kalkaji Mandir,Kalkaji Mandir,KJMD,4,0,September,Entry
3,2024-09-01,LINE06,Gobind Puri,Govindpuri,GDPI,4,0,September,Entry
4,2024-09-01,LINE06,Okhla,Harkesh Nagar Okhla,OK,4,0,September,Entry



Weather sample:


,time,temperature_2m_c,relative_humidity_2m_pct,apparent_temperature_c,precipitation_mm,rain_mm,weather_code_wmo,surface_pressure_hpa,cloud_cover_pct,visibility_m,wind_speed_10m_kmh
0,2024-09-01 00:00:00,27.2,90,33.4,0.0,0.0,0,980.6,4,24140.0,5.1
1,2024-09-01 01:00:00,27.1,89,33.0,0.0,0.0,0,980.0,2,24140.0,5.9
2,2024-09-01 02:00:00,26.7,91,32.4,0.0,0.0,0,979.7,2,24140.0,6.9
3,2024-09-01 03:00:00,26.4,91,32.0,0.0,0.0,0,979.8,1,24140.0,7.2
4,2024-09-01 04:00:00,26.2,92,31.8,0.0,0.0,0,980.0,1,24140.0,7.2


## 2. Aggregate the station-level counts to citywide hourly ridership

We keep `Entry` and `Exit` separate, because the source data records them independently and they should not be merged into a single total before we confirm a consistent modeling definition.

The aggregation below sums the station counts by date, hour, and direction.

In [ ]:
citywide_ridership = (
    metro.groupby(['businessday', 'hour', 'direction'], as_index=False)['count']
    .sum()
    .rename(columns={'count': 'ridership_count'})
)

citywide_ridership['businessday'] = pd.to_datetime(citywide_ridership['businessday'])
citywide_ridership['time'] = (
    citywide_ridership['businessday'].dt.normalize() +
    pd.to_timedelta(citywide_ridership['hour'], unit='h')
)

print('Citywide ridership rows:', len(citywide_ridership))
print('Duplicate date/hour/direction keys:', citywide_ridership.duplicated(subset=['businessday', 'hour', 'direction']).sum())
print('Time range:', citywide_ridership['time'].min(), 'to', citywide_ridership['time'].max())
print('Missing ridership values:', citywide_ridership['ridership_count'].isna().sum())
print('\nCitywide ridership by direction:')
display(citywide_ridership['direction'].value_counts().to_frame('rows'))

citywide_ridership.head()

## 3. Validate the aggregation against the source data

The raw metro data records hours 00:00 and then 04:00 onward, with no 01:00–03:00 rows. This is consistent with the operational schedule in the source workbook and should be preserved rather than filled with guessed values.

The citywide aggregation therefore does not produce a full 24-hour series; it produces one row per observed operating hour and direction.

In [ ]:
print('Hours present in source metro data:', sorted(metro['hour'].unique()))
print('\nCount per hour across the source data:')
display(metro.groupby('hour').size().sort_index().rename('records_per_hour').to_frame())

print('\nHours with any ridership on a sample date:')
example_date = citywide_ridership['businessday'].min().date()
example = citywide_ridership[citywide_ridership['businessday'].dt.date == example_date].sort_values(['direction', 'hour'])
display(example.head(20))

## 4. Merge with weather while preserving Entry and Exit separately

This merge is performed on the hourly timestamp. We validate both the outer merge (to identify unmatched weather hours) and the inner merge (to create the aligned modeling-ready dataset only for hours that are present in both sources).

In [ ]:
# Outer merge for audit: shows missing hours and the real match coverage
outer = citywide_ridership.merge(weather[['time']], on='time', how='outer', indicator=True)
print('Outer merge coverage:')
display(outer['_merge'].value_counts().to_frame('rows'))
print('Unmatched weather rows (right_only):', (outer['_merge'] == 'right_only').sum())
print('Unmatched ridership rows (left_only):', (outer['_merge'] == 'left_only').sum())

# Inner merge: only rows that are aligned across both data sources.
merged = citywide_ridership.merge(weather, on='time', how='inner', validate='many_to_one', sort=True)
merged = merged.sort_values(['time', 'direction']).reset_index(drop=True)

print('\nAligned inner merge rows:', len(merged))
print('Duplicate time+direction keys:', merged.duplicated(subset=['time', 'direction']).sum())
print('Missing ridership values after merge:', merged['ridership_count'].isna().sum())
print('Missing weather values after merge:', merged[[c for c in merged.columns if c not in {'time', 'businessday', 'hour', 'direction'}]].isna().sum().sum())
print('Unique aligned timestamps:', merged['time'].nunique())

merged.head()

In [ ]:
# Keep the aligned hourly dataset in a clean, model-ready order.
merged = merged[
    [
        'time', 'businessday', 'hour', 'direction', 'ridership_count',
        'temperature_2m_c', 'relative_humidity_2m_pct', 'apparent_temperature_c',
        'precipitation_mm', 'rain_mm', 'weather_code_wmo', 'surface_pressure_hpa',
        'cloud_cover_pct', 'visibility_m', 'wind_speed_10m_kmh'
    ]
].copy()

merged.to_csv(output_path, index=False)
print(f'Created merged dataset at: {output_path}')
print(f'Final row count: {len(merged):,}')
print(f'Time range: {merged["time"].min()} to {merged["time"].max()}')
print('Direction counts:')
display(merged['direction'].value_counts().to_frame('rows'))

## 5. Data-quality summary before finalizing feature engineering

The merge check passes with the following observations:

- `count` is a station-level hourly flow value, not a citywide series.
- Citywide ridership is valid when summed across stations by `businessday`, `hour`, and `direction`.
- The source metro dataset has 00:00 and 04:00–23:00 values; 01:00–03:00 are absent, which reflects real operating hours rather than a formatting bug.
- The aligned inner merge contains 7,602 rows: 3,801 Entry rows and 3,801 Exit rows.
- The outer merge confirms 543 weather-only hours fall outside the observed metro operating schedule; they are not data errors but unmatched non-operational hours.
- No duplicate merge keys and no missing merged values were found.

This is the correct checkpoint before any lag or rolling feature creation.

## Stage 1: Temporal feature preparation

This stage adds only the time-based features required before modeling. It keeps Entry and Exit separate, preserves the source overnight gap from 01:00–03:00, and avoids any lag or rolling feature creation.

In [ ]:
import numpy as np

source_path = project_root / 'DataProcessed' / 'metro_weather_hourly_merged.csv'
output_path = project_root / 'DataProcessed' / 'metro_weather_temporal_features.csv'

merged = pd.read_csv(source_path, parse_dates=['time'])
df = merged.copy()

df['time'] = pd.to_datetime(df['time'])
df['hour'] = df['time'].dt.hour
df['day_of_week'] = df['time'].dt.dayofweek
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['month'] = df['time'].dt.month

# Peak-hour windows are derived from the existing ridership EDA: morning peak 08:00–10:00 and evening peak 18:00–19:00.
df['is_peak_hour'] = df['hour'].isin([8, 9, 10, 18, 19]).astype(int)

df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

print(f'Input rows: {len(merged):,}')
print(f'Output rows: {len(df):,}')
print('Missing hours in source schedule:', sorted(set(range(24)) - set(df['hour'].unique())))
print('Hour range:', df['hour'].min(), 'to', df['hour'].max())
print('Day-of-week range:', df['day_of_week'].min(), 'to', df['day_of_week'].max())
print('Weekend values:', sorted(df['is_weekend'].unique()))
print('Month range:', df['month'].min(), 'to', df['month'].max())
print('Peak-hour values:', sorted(df['is_peak_hour'].unique()))
print('Peak-hour windows used:', {8, 9, 10, 18, 19})
print('Duplicate timestamp+direction keys:', df.duplicated(subset=['time', 'direction']).sum())
print('Missing values:', df.isna().sum().to_dict())

assert len(df) == len(merged)
assert df.duplicated(subset=['time', 'direction']).sum() == 0
assert df['hour'].between(0, 23).all()
assert df['day_of_week'].between(0, 6).all()
assert df['is_weekend'].isin([0, 1]).all()
assert df['month'].between(1, 12).all()
assert df['is_peak_hour'].isin([0, 1]).all()
assert set(range(24)) - set(df['hour'].unique()) == {1, 2, 3}
assert df.isna().sum().sum() == 0

# Preserve the source overnight gap and do not impute missing hours.
df.to_csv(output_path, index=False)
print(f'Saved temporal feature dataset to: {output_path}')

This stage is complete. The temporal feature dataset preserves the original timing pattern and keeps Entry and Exit separate. Lags, rolling features, and modeling are intentionally deferred until the feature-engineering checkpoint after this validation pass.

## Stage 2: Rainfall and weather feature preparation

This stage adds rainfall and weather-only features to the validated temporal dataset. It does not create ridership lags, interactions, or models. Historical rainfall features are built using exact timestamps and the real 01:00–03:00 gaps are preserved.

In [ ]:
weather_path = project_root / 'DataProcessed' / 'weather_hourly_cleaned.csv'
metro_path = project_root / 'DataProcessed' / 'metro_weather_temporal_features.csv'
output_path = project_root / 'DataProcessed' / 'metro_weather_weather_features.csv'

weather = pd.read_csv(weather_path, parse_dates=['time']).sort_values('time').reset_index(drop=True)
metro = pd.read_csv(metro_path, parse_dates=['time']).sort_values(['time', 'direction']).reset_index(drop=True)

assert weather['time'].is_unique
assert weather['time'].diff().dropna().eq(pd.Timedelta(hours=1)).all()
assert not metro.duplicated(subset=['time', 'direction']).any()

# Engineer rainfall history on the complete weather timeline, before joining metro rows.
weather_features = weather[['time', 'rain_mm']].copy()
bins = [-np.inf, 0, 2.5, 7.6, np.inf]
labels = ['No Rain', 'Light Rain', 'Moderate Rain', 'Heavy Rain']
weather_features['rainfall_intensity'] = pd.cut(
    weather_features['rain_mm'],
    bins=bins,
    labels=labels,
    right=True,
    include_lowest=False
).astype(str)
weather_features['is_raining'] = (weather_features['rain_mm'] > 0).astype(int)

for lag in [1, 2, 3]:
    lag_lookup = weather[['time', 'rain_mm']].rename(columns={'rain_mm': f'rainfall_lag_{lag}h'}).copy()
    lag_lookup['time'] = lag_lookup['time'] + pd.Timedelta(hours=lag)
    weather_features = weather_features.merge(lag_lookup, on='time', how='left', validate='one_to_one')

weather_rain = weather.set_index('time')['rain_mm']
for window in [3, 6, 12, 24]:
    weather_features[f'rainfall_sum_{window}h'] = (
        weather_rain.rolling(f'{window}h', closed='left', min_periods=window)
        .sum()
        .to_numpy()
    )

feature_columns = [
    'rainfall_intensity', 'is_raining',
    'rainfall_lag_1h', 'rainfall_lag_2h', 'rainfall_lag_3h',
    'rainfall_sum_3h', 'rainfall_sum_6h', 'rainfall_sum_12h', 'rainfall_sum_24h'
]
df = metro.merge(
    weather_features[['time', *feature_columns]],
    on='time',
    how='left',
    validate='many_to_one'
).sort_values(['time', 'direction']).reset_index(drop=True)

assert len(df) == len(metro)
assert df[['time', 'direction']].equals(metro[['time', 'direction']])
assert not df.duplicated(subset=['time', 'direction']).any()
assert df['rainfall_intensity'].notna().all()
assert df['is_raining'].isin([0, 1]).all()

# Verify overnight history at the first metro hour after the daily operating gap.
weather_by_time = weather.set_index('time')['rain_mm']
for sample_time in [pd.Timestamp('2024-09-01 04:00:00'), pd.Timestamp('2024-09-02 04:00:00')]:
    sample = df.loc[df['time'].eq(sample_time)]
    assert len(sample) == 2 and set(sample['direction']) == {'Entry', 'Exit'}
    for lag in [1, 2, 3]:
        expected = weather_by_time.loc[sample_time - pd.Timedelta(hours=lag)]
        assert sample[f'rainfall_lag_{lag}h'].eq(expected).all()
    for window in [3, 6, 12, 24]:
        start = sample_time - pd.Timedelta(hours=window)
        expected = weather_by_time.loc[(weather_by_time.index >= start) & (weather_by_time.index < sample_time)].sum()
        actual = sample[f'rainfall_sum_{window}h']
        if start < weather_by_time.index.min():
            assert actual.isna().all()
        else:
            assert actual.eq(expected).all()
    print(f'Manual check at {sample_time}:')
    print(sample[['direction', 'rainfall_lag_1h', 'rainfall_lag_2h', 'rainfall_lag_3h', 'rainfall_sum_3h', 'rainfall_sum_6h', 'rainfall_sum_12h', 'rainfall_sum_24h']].to_string(index=False))

missing_by_column = df.isna().sum()
print('Missing values by column:')
print(missing_by_column.to_string())
print('Rows:', len(df), '| duplicate time-direction keys:', df.duplicated(['time', 'direction']).sum())

df.to_csv(output_path, index=False)
print(f'Saved weather-feature dataset to: {output_path}')

Stage 2 is complete. The new dataset contains rainfall intensity, rain indicator, exact lagged rainfall, and rolling rainfall totals while preserving the real hourly schedule and keeping Entry and Exit separate.

## Stage 3: Ridership history features

Create direction-specific ridership lags and rolling averages from `ridership_count`. Lags are matched at exact timestamps. Rolling averages use preceding clock-time windows only and exclude the current timestamp. The 3h and 6h means require complete hourly windows; the 24h mean averages available historical observations and requires at least one. Missing metro hours are never treated as zero. The Stage 2 dataset is preserved, and the result is saved as `metro_weather_features_final.csv`.

In [5]:
import numpy as np

source_path = project_root / 'DataProcessed' / 'metro_weather_weather_features.csv'
output_path = project_root / 'DataProcessed' / 'metro_weather_features_final.csv'
target = 'ridership_count'

source = pd.read_csv(source_path, parse_dates=['time'])
assert target in source.columns, f'Required ridership target not found: {target}'
assert not source.duplicated(['time', 'direction']).any()

df = source.sort_values(['time', 'direction']).reset_index(drop=True).copy()
for lag in [1, 2, 3, 24]:
    lag_lookup = df[['time', 'direction', target]].copy()
    lag_lookup = lag_lookup.rename(columns={target: f'ridership_lag_{lag}h'})
    lag_lookup['time'] = lag_lookup['time'] + pd.Timedelta(hours=lag)
    df = df.merge(lag_lookup, on=['time', 'direction'], how='left', validate='one_to_one')

for window in [3, 6, 24]:
    feature = f'ridership_rolling_mean_{window}h'
    df[feature] = np.nan
    min_observations = 1 if window == 24 else window
    for direction, group in df.groupby('direction', sort=False):
        ordered = group.sort_values('time')
        rolling_mean = ordered.set_index('time')[target].rolling(
            f'{window}h', closed='left', min_periods=min_observations
        ).mean()
        df.loc[ordered.index, feature] = rolling_mean.to_numpy()

df = df.sort_values(['time', 'direction']).reset_index(drop=True)
new_features = [
    'ridership_lag_1h', 'ridership_lag_2h', 'ridership_lag_3h', 'ridership_lag_24h',
    'ridership_rolling_mean_3h', 'ridership_rolling_mean_6h', 'ridership_rolling_mean_24h'
]
assert len(df) == len(source)
assert not df.duplicated(['time', 'direction']).any()
assert set(source.columns).issubset(df.columns)

# Manually check exact lags and the preceding 24-hour window at 04:00 after the overnight gap.
sample_time = pd.Timestamp('2024-09-02 04:00:00')
sample_rows = df.loc[df['time'].eq(sample_time)]
source_by_key = source.set_index(['time', 'direction'])[target]
for _, row in sample_rows.iterrows():
    direction = row['direction']
    for lag in [1, 2, 3, 24]:
        key = (sample_time - pd.Timedelta(hours=lag), direction)
        expected = source_by_key.loc[key] if key in source_by_key.index else np.nan
        actual = row[f'ridership_lag_{lag}h']
        assert (pd.isna(expected) and pd.isna(actual)) or actual == expected
    prior = source.loc[
        source['direction'].eq(direction)
        & source['time'].ge(sample_time - pd.Timedelta(hours=24))
        & source['time'].lt(sample_time), target
    ]
    expected_mean = prior.mean() if not prior.empty else np.nan
    actual_mean = row['ridership_rolling_mean_24h']
    assert (pd.isna(expected_mean) and pd.isna(actual_mean)) or np.isclose(actual_mean, expected_mean)

print(f'Manual check at {sample_time}:')
print(sample_rows[['direction', *new_features]].to_string(index=False))
print('Rows:', len(df), '| duplicate time-direction keys:', df.duplicated(['time', 'direction']).sum())
print('Missing values in new features:')
print(df[new_features].isna().sum().to_string())
df.to_csv(output_path, index=False)
print(f'Saved final feature dataset to: {output_path}')

Manual check at 2024-09-02 04:00:00:
direction  ridership_lag_1h  ridership_lag_2h  ridership_lag_3h  ridership_lag_24h  ridership_rolling_mean_3h  ridership_rolling_mean_6h  ridership_rolling_mean_24h
    Entry               NaN               NaN               NaN                1.0                        NaN                        NaN               106074.761905
     Exit               NaN               NaN               NaN                0.0                        NaN                        NaN               109386.333333
Rows: 7602 | duplicate time-direction keys: 0
Missing values in new features:
ridership_lag_1h               364
ridership_lag_2h               726
ridership_lag_3h              1088
ridership_lag_24h               42
ridership_rolling_mean_3h     1088
ridership_rolling_mean_6h     2174
ridership_rolling_mean_24h       2
Saved final feature dataset to: /Users/shivaya/Desktop/Desktop - Utkarsh’s MacBook Pro/Research/DelhiMetroResearch/DataProcessed/metro_weather_fe

## Stage 4: Model preparation and validation

`ridership_count` is the target, with Entry and Exit retained as separate directional series. Predictors exclude raw timestamps/date identifiers, the current target, and redundant encodings: `precipitation_mm` duplicates `rain_mm`; `rainfall_intensity` and `is_raining` are derived from `rain_mm`; raw `hour`, `day_of_week`, and `is_weekend` are omitted in favor of cyclical time features. Exact historical ridership lags and prior-window means are retained; no future target values enter the predictors. `time` remains in exported partitions only for audit, not as a model input. Same-hour weather is treated as available at prediction time; operational forecasting should supply weather forecasts for that timestamp.

Rows with missing model inputs are excluded, never filled with zero. This is primarily due to unavailable exact-hour lags and incomplete required 3h/6h history windows at metro off-hours or the start of the series. Eligible timestamps are split chronologically 70%/15%/15%, with both directions assigned together. No models are trained in this stage.

In [6]:
source_path = project_root / 'DataProcessed' / 'metro_weather_features_final.csv'
output_dir = project_root / 'DataProcessed'

stage4_source = pd.read_csv(source_path, parse_dates=['time'])
target = 'ridership_count'
assert target in stage4_source.columns
assert not stage4_source.duplicated(['time', 'direction']).any()
assert stage4_source['direction'].isin(['Entry', 'Exit']).all()
assert stage4_source['precipitation_mm'].equals(stage4_source['rain_mm'])

model_features = [
    'direction',
    'temperature_2m_c', 'relative_humidity_2m_pct', 'apparent_temperature_c',
    'rain_mm', 'weather_code_wmo', 'surface_pressure_hpa', 'cloud_cover_pct',
    'visibility_m', 'wind_speed_10m_kmh',
    'month', 'is_peak_hour', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'rainfall_lag_1h', 'rainfall_lag_2h', 'rainfall_lag_3h',
    'rainfall_sum_3h', 'rainfall_sum_6h', 'rainfall_sum_12h', 'rainfall_sum_24h',
    'ridership_lag_1h', 'ridership_lag_2h', 'ridership_lag_3h', 'ridership_lag_24h',
    'ridership_rolling_mean_3h', 'ridership_rolling_mean_6h', 'ridership_rolling_mean_24h'
]
excluded_identifiers = ['time', 'businessday']
assert target not in model_features
assert not set(excluded_identifiers).intersection(model_features)
assert not any('future' in column.lower() for column in model_features)
assert set(model_features + [target, *excluded_identifiers]).issubset(stage4_source.columns)

complete_rows = stage4_source[model_features + [target]].notna().all(axis=1)
excluded_rows = stage4_source.loc[~complete_rows]
prepared = stage4_source.loc[complete_rows].sort_values(['time', 'direction']).copy()
assert prepared.groupby('time')['direction'].nunique().eq(2).all()

# Split the eligible timestamps, not individual rows, so directions stay together.
eligible_times = prepared['time'].drop_duplicates().sort_values().reset_index(drop=True)
train_end = int(round(len(eligible_times) * 0.70))
validation_end = int(round(len(eligible_times) * 0.85))
time_splits = {
    'train': set(eligible_times.iloc[:train_end]),
    'validation': set(eligible_times.iloc[train_end:validation_end]),
    'test': set(eligible_times.iloc[validation_end:])
}

split_frames = {}
for split_name, split_times in time_splits.items():
    split = prepared.loc[prepared['time'].isin(split_times)].sort_values(['time', 'direction']).copy()
    X_split = split[model_features]
    y_split = split[target]
    assert X_split.index.equals(y_split.index)
    assert X_split.notna().all().all() and y_split.notna().all()
    assert not split.duplicated(['time', 'direction']).any()
    split_frames[split_name] = split[['time', *model_features, target]]
    split_frames[split_name].to_csv(output_dir / f'metro_weather_{split_name}_prepared.csv', index=False)

train_times = sorted(time_splits['train'])
validation_times = sorted(time_splits['validation'])
test_times = sorted(time_splits['test'])
assert train_times[-1] < validation_times[0] < validation_times[-1] < test_times[0]
assert sum(len(frame) for frame in split_frames.values()) == len(prepared)
assert not (time_splits['train'] & time_splits['validation'])
assert not (time_splits['train'] & time_splits['test'])
assert not (time_splits['validation'] & time_splits['test'])

print('Target:', target, '| direction remains a model feature')
print('Model feature count:', len(model_features))
print('Dropped redundant/identifier columns:', excluded_identifiers + ['precipitation_mm', 'rainfall_intensity', 'is_raining', 'hour', 'day_of_week', 'is_weekend'])
print('Rows excluded for missing model inputs:', len(excluded_rows))
print('Missing feature counts among excluded rows:')
print(excluded_rows[model_features].isna().sum().loc[lambda counts: counts.gt(0)].to_string())
print('Eligible rows:', len(prepared), '| eligible timestamps:', len(eligible_times))
for split_name, frame in split_frames.items():
    print(f"{split_name}: rows={len(frame)}, timestamps={frame['time'].nunique()}, from={frame['time'].min()} to={frame['time'].max()}")
print('Split feature/target missing values:')
for split_name, frame in split_frames.items():
    print(split_name, 'X missing=', int(frame[model_features].isna().sum().sum()), 'y missing=', int(frame[target].isna().sum()))
print('All split duplicate time-direction keys:', sum(int(frame.duplicated(['time', 'direction']).sum()) for frame in split_frames.values()))

Target: ridership_count | direction remains a model feature
Model feature count: 30
Dropped redundant/identifier columns: ['time', 'businessday', 'precipitation_mm', 'rainfall_intensity', 'is_raining', 'hour', 'day_of_week', 'is_weekend']
Rows excluded for missing model inputs: 2202
Missing feature counts among excluded rows:
rainfall_lag_1h                  2
rainfall_lag_2h                  2
rainfall_lag_3h                  2
rainfall_sum_3h                  2
rainfall_sum_6h                  6
rainfall_sum_12h                18
rainfall_sum_24h                42
ridership_lag_1h               364
ridership_lag_2h               726
ridership_lag_3h              1088
ridership_lag_24h               42
ridership_rolling_mean_3h     1088
ridership_rolling_mean_6h     2174
ridership_rolling_mean_24h       2
Eligible rows: 5400 | eligible timestamps: 2700
train: rows=3780, timestamps=1890, from=2024-09-02 00:00:00 to=2025-01-05 23:00:00
validation: rows=810, timestamps=405, from=2025-01-